In [ ]:
import zarr, s3fs
import xarray as xr
from pathlib import Path

Read from S3 bucket directly

In [ ]:
eodc_s3 = s3fs.S3FileSystem(
    key="",
    secret="",
    client_kwargs={
        "endpoint_url": "https://objects.eodc.eu"
    })
zarr_stores = eodc_s3.ls("destine-climate-dt/IFS-NEMO-down")
zarr_stores

Read from disc

In [ ]:
root_path = Path("/Users/christophreimer/datapool/scratch/IFS-NEMO-down")
zrec = []
for zs in root_path.glob("*.zarr"):
    zrec.append({"file": zs, "datetime": xr.open_zarr(zs)["datetimes"].min().values.item()})
print(f"{len(zrec)} zarr stores found.")

In [ ]:
from datetime import datetime
records_sorted = sorted(
    zrec,
    key=lambda r: r["datetime"])
records_sorted
zstores_sorted = [zfd["file"] for zfd in records_sorted]

In [ ]:
records_sorted

get the first and last datetimes from the sorted zarr stores

In [ ]:
min_date = xr.open_dataset(zstores_sorted[0]).coords["datetimes"].values.astype("datetime64[s]").min()
max_date = xr.open_dataset(zstores_sorted[-1]).coords["datetimes"].values.astype("datetime64[s]").max()

In [ ]:
datetime_samples = xr.date_range(start=min_date, end=max_date, freq="1h")
n_datetime_samples = datetime_samples.size
n_datetime_samples

get number healpix grid points

In [ ]:
n_grid_points = xr.open_dataset(zstores_sorted[0]).coords["points"].values.size
n_grid_points

In [ ]:
dummy_ds = xr.open_dataset(zstores_sorted[0])
healpix_lat = dummy_ds.coords["latitude"].values
healpix_lon = dummy_ds.coords["longitude"].values

create an empty zarr store where all the data gets written to

In [ ]:
import dask.array as da
import numpy as np

climatedt_zarr = Path("/Users/christophreimer/datapool/scratch/IFS-NEMO-CMIP6-ts.zarr")
points_chunksize = 100

data_arr = da.zeros((n_datetime_samples, n_grid_points), chunks=(n_datetime_samples, points_chunksize), dtype=np.float64)
ds = xr.Dataset(data_vars={"2t": (("datetimes", "points"), data_arr),
                           "tp": (("datetimes", "points"), data_arr)},
                coords={"datetimes": (("datetimes"), datetime_samples),
                        "points":(("points"), np.arange(n_grid_points)),
                        "latitude": (("points"), healpix_lat),
                        "longitude": (("points"), healpix_lon)})
ds.to_zarr(climatedt_zarr, mode="w", compute=False,
           encoding={
               "datetimes": {
                   "chunks": n_datetime_samples
               },
               "points": {
                   "chunks": n_grid_points
               },
               "latitude": {
                   "chunks": n_grid_points
               },
               "longitude": {
                   "chunks": n_grid_points
               }
           })

now we want to fill each chunk with the corresponding data.

In [ ]:
chunk_pos_start = np.arange(0, n_grid_points, points_chunksize)
chunk_pos_end = np.append(np.arange(points_chunksize-1, n_grid_points, points_chunksize), [n_grid_points-1])

In [ ]:
dtcube = zarr.open(climatedt_zarr, mode="a")
zarr_2t = dtcube["2t"]
zarr_tp = dtcube["tp"]

In [ ]:
for p in zstores_sorted:
    ds = xr.open_zarr(p)
    datetime_coord = ds.coords["datetimes"].astype("datetime64[s]")
    ds = ds.assign_coords(datetimes=datetime_coord)
    time_slice = slice(datetime_samples.get_loc(datetime_coord.min().values),
                    datetime_samples.get_loc(datetime_coord.max().values))

    zarr_2t[time_slice, :] = ds["2t"].data
    zarr_tp[time_slice, :] = ds["tp"].data

Verify that data was written correctly.

In [ ]:
zstore_timedim = xr.open_dataset(climatedt_zarr, chunks="auto")
zstore_timedim["tp"].isel(points=200).plot()